In [34]:
from dotenv import load_dotenv
from langchain_classic.agents.xml.prompt import agent_instructions
from langchain_core.messages import HumanMessage

load_dotenv()#本地.env一定会要UV add python-dotenv
#再初始化后进行
import os
from langchain.chat_models import  init_chat_model
base_url=os.environ["IMAGE_BASE_URL"]#加载环境变量里的这两个值
api_key=os.environ["API_KEY"]

model=init_chat_model(
    model="qwen3.5-plus",#模型名称选择阿里云百炼平台支持的
    model_provider="openai",#借用openai这个通道，适用于langchain不支持的模型，需要指定模型的提供者
    api_key=api_key,
    base_url=base_url,

)


In [35]:
#创建 Agent
from langchain.agents import create_agent
agent=create_agent(model=model)

In [38]:
#准备1多模态消息
# message = {
#     "role": "user",
#     "content": [
#         {"type": "text", "text": "Describe the content of this image."},
#         {"type": "image", "url": "https://www.bing.com/th/id/OIP.OFMlQdJWRGUjT2PNEWN00AHaEK?w=312&h=180&c=7&r=0&o=7&dpr=1.5&pid=1.7&rm=3"},
#     ]
# }
#现在又变成字典了,放入数组，因为humanmessage本来就代表content
message=HumanMessage([
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [39]:
#调用，流式
stream=agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk,metadata in stream:
    if chunk.content:
        print(chunk.content,end="",flush=True)

This image captures a serene and heartwarming moment between a woman and her dog on a sandy beach, likely during the "golden hour" of sunset or sunrise given the warm, bright lighting.

**The Subjects:**
*   **The Woman:** She is sitting on the sand with her legs bent, facing the dog. She has long dark hair and is wearing a blue and white plaid button-down shirt over dark pants that are rolled up at the ankles. She is barefoot and smiling warmly at the dog. On her left wrist, she wears a white watch.
*   **The Dog:** It appears to be a Golden Retriever or Labrador with light golden fur. The dog is sitting on its hind legs, facing the woman. It is wearing a blue harness with a colorful pattern. The dog has its right front paw raised, meeting the woman's hand in a "shake" or high-five gesture.

**The Setting:**
*   **The Beach:** They are sitting on textured sand with visible footprints. A red leash lies on the sand near the dog's paws.
*   **The Background:** Behind them, the ocean stre

In [40]:
from ipywidgets import FileUpload
from IPython.display import display
uploader=FileUpload(accept='*',multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [41]:
print(uploader.value)

({'name': 'QQ图片20240313225356.jpg', 'type': 'image/jpeg', 'size': 87723, 'content': <memory at 0x0000018EB4356BC0>, 'last_modified': datetime.datetime(2024, 3, 13, 14, 49, 51, 145000, tzinfo=datetime.timezone.utc)},)


In [43]:
#读取图片，转为base64字符串
import base64
#获取第一个也是唯一一个上传的文件
uploaded_file=uploader.value[0]#从上传组件里取图片
#获取其内存视图,拿到具体内容
content_mv=uploaded_file["content"]
#转换内存视图->字节,将内容转成字节
img_bytes=bytes(content_mv)
#将自己进行base64编码
img_b64=base64.b64encode(img_bytes).decode("utf-8")

In [44]:
#组织多模态消息
# message = {
#     "role": "user",
#     "content": [
#         {"type": "text", "text": "Describe the content of this image."},
#         {
#             "type": "image",
#             "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
#             "mime_type": "image/jpeg",
#         },
#     ]
# }
multimodel_question=HumanMessage(content=[
    {"type":"image",
     "base64": img_b64,
     "mime_type":"image/jpeg"
     },
    {"type":"text","text":"Describe the content of this image."},

])
#两步合成一步
for chunk,metadata in agent.stream(
        {"messages": [multimodel_question]},
    stream_mode="messages"
):
    print(chunk.content,end="",flush=True)


This is a soft, warm-toned digital illustration of an anime character, recognizable as **Sakura Kinomoto** from the series *Cardcaptor Sakura*.

Here are the specific details:

*   **Character Appearance:** She has short, layered brown hair with lighter, blonde-highlighted tips that frame her face. Her eyes are large and expressive, colored a soft hazel-green with bright highlights. She has a gentle, slightly flushed complexion.
*   **Action:** She is holding the **Sealing Key** (her magic wand in its key form) delicately between her fingers, holding it right up against her lips as if she is about to kiss it or is shyly hiding behind it. The key features a pink circle with a yellow star in the center and small white wings.
*   **Clothing:** She is wearing a dark (likely black or deep green) turtleneck sweater underneath a traditional Japanese sailor-style school uniform (seifuku). The uniform has a white collar with red stripes and matching stripes on the cuffs.
*   **Art Style:** The 

In [46]:
#组织多模态消息
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {
            "type": "image",
            "base64": img_b64,
            "mime_type": "image/jpeg",
        },
    ]
}
# multimodel_question=HumanMessage(content=[
#     {"type":"image",
#      "base64": img_b64,
#      "mime_type":"image/jpeg"
#      },
#     {"type":"text","text":"Describe the content of this image."},
#
# ])
#两步合成一步
for chunk,metadata in agent.stream(
        {"messages": [message]},
    stream_mode="messages"
):
    print(chunk.content,end="",flush=True)


This is a close-up, anime-style illustration of a young girl, who appears to be the character Sakura Kinomoto from the series *Cardcaptor Sakura*.

**Key details include:**

*   **Character:** She has short, tousled light brown hair with bangs framing her face. Her large eyes are a mix of green and hazel, reflecting light softly. She has a gentle, slightly flushed expression.
*   **Action:** She is holding a magical-looking key up to her lips, as if she is about to kiss it or is holding it delicately near her mouth.
*   **The Object:** The key she is holding is distinctive—it features a pink circular top with a gold star in the center and small white wings on either side. This is recognizable as the "Sealing Key" from the series.
*   **Clothing:** She is wearing a dark school uniform with a white sailor-style collar that has red stripes along the edge. Underneath the uniform, she wears a black turtleneck.
*   **Lighting and Style:** The image has a warm, golden lighting effect, suggest